# **MODEL PREDICTION**

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from collections import Counter
import datetime
import joblib

# 1. PERSIAPAN DATASET
print("=== Memuat dataset dari final_stress_dataset.csv ===")
df = pd.read_csv('final_stress_dataset.csv', delimiter=';')

# Pemetaan otomatis kolom kategorikal objek ke numerik
if 'daily_pressure' in df.columns and df['daily_pressure'].dtype == 'object':
    print("[INFO] Mengonversi kolom kategorikal 'daily_pressure' ke numerik...")
    pressure_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
    df['daily_pressure'] = df['daily_pressure'].map(pressure_mapping)

TARGET_COLUMN = 'stress_level'
X_df = df.drop(columns=[TARGET_COLUMN])
y_text = df[TARGET_COLUMN].values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

num_classes = len(label_encoder.classes_)
print(f"Jumlah Fitur (Input): {X_df.shape[1]}")
print(f"Jumlah Kelas (Target): {num_classes} -> {list(label_encoder.classes_)}")

# Simpan nama-nama kolom untuk penyeimbangan data sintetis
feature_names = X_df.columns.tolist()
X = X_df.astype(float).values

# 2. SPLIT DAN PENAMBAHAN DATA SINTETIS (SMOTE BEST PRACTICE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nDistribusi kelas SEBELUM SMOTE (train): {Counter(y_train)}")
print(f"Shape Data Train SEBELUM SMOTE: {X_train.shape}")

# SMOTE pada data latih mentah agar skala fitur tetap riil
smote = SMOTE(random_state=42, k_neighbors=2)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# shape tepat setelah fungsi SMOTE dijalankan ---
print(f"\n[INFO] Shape Data Latih TEPAT SETELAH SMOTE: {X_train_resampled.shape}")

# Pembulatan fitur diskrit/kategorikal pada data sintetis hasil SMOTE
X_train_resampled_df = pd.DataFrame(X_train_resampled, columns=feature_names)
discrete_columns = [
    'gender', 'quality_of_sleep', 'daily_pressure', 'bmi_category', 'heart_rate', 'daily_steps'
] + [col for col in feature_names if col.startswith('q')]

for col in discrete_columns:
    if col in X_train_resampled_df.columns:
        X_train_resampled_df[col] = X_train_resampled_df[col].round().astype(int)

X_train_resampled = X_train_resampled_df.values

# shape final pasca pembulatan
print(f"Shape Data Latih SETELAH Pembulatan Diskrit: {X_train_resampled.shape}")
print(f"Distribusi kelas SETELAH SMOTE (train): {Counter(y_train_resampled)}")

# Standardisasi fitur (Scaling) setelah data sintetis seimbang dan rapi
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

num_features = X_train_scaled.shape[1]

# Konversi target ke matriks kategorikal (One-Hot Encoding)
y_train_cat = tf.keras.utils.to_categorical(y_train_resampled, num_classes=num_classes)
y_test_cat  = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)


# 3. KUSTOM LAYER KEPATUHAN SERIALISASI KERAS
@tf.keras.utils.register_keras_serializable(package="CustomLayers")
class StressFeatureExtractor(tf.keras.layers.Layer):
    def __init__(self, units=32, **kwargs):
        super(StressFeatureExtractor, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1], self.units),
                                 initializer='random_normal', trainable=True, name='w_custom')
        self.b = self.add_weight(shape=(self.units,),
                                 initializer='zeros', trainable=True, name='b_custom')
        super(StressFeatureExtractor, self).build(input_shape)

    def call(self, inputs):
        return tf.nn.relu(tf.matmul(inputs, self.w) + self.b)

    def get_config(self):
        config = super(StressFeatureExtractor, self).get_config()
        config.update({"units": self.units})
        return config


# 4. MEMBANGUN ARSITEKTUR FUNCTIONAL MODEL
def build_functional_model(input_dim, output_dim):
    inputs = tf.keras.Input(shape=(input_dim,), name=f"input_{input_dim}_features")

    x = StressFeatureExtractor(units=64, name="custom_feature_extractor")(inputs)
    x = tf.keras.layers.Dense(128, activation='relu', name="dense_dense_1")(x)
    x = tf.keras.layers.Dropout(0.3, name="dropout_1")(x)

    x = tf.keras.layers.Dense(64, activation='relu', name="dense_dense_2")(x)
    x = tf.keras.layers.Dropout(0.2, name="dropout_2")(x)

    x = tf.keras.layers.Dense(32, activation='relu', name="dense_dense_3")(x)

    # Menambahkan argumen name secara eksplisit pada output layer
    outputs = tf.keras.layers.Dense(output_dim, activation='softmax', name="output_stress_level")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Stress_Prediction_Model")
    return model

model = build_functional_model(input_dim=num_features, output_dim=num_classes)

print("\n=== RINGKASAN ARSITEKTUR MODEL ===")
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


# 5. TRAINING PROCESS (PROSES PELATIHAN)
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=15, restore_best_weights=True, mode='max'
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
)

print("\nMemulai proses training...")
history = model.fit(
    X_train_scaled, y_train_cat,
    epochs=100,
    batch_size=32,
    validation_data=(X_test_scaled, y_test_cat),
    callbacks=[tensorboard_callback, early_stop, lr_scheduler],
    verbose=1
)


# 6. EVALUASI AKHIR DAN EKSPOR ARTIFAK
loss, accuracy = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"\n--- Evaluasi Akhir ---")
print(f"Akurasi Test: {accuracy * 100:.2f}%")

y_pred = np.argmax(model.predict(X_test_scaled), axis=1)
print("\n--- Classification Report ---")
target_names_display = ['High (0)', 'Low (1)', 'Medium (2)']
print(classification_report(y_test, y_pred, target_names=target_names_display, zero_division=0))

# Simpan berkas
model_path = "stress_prediction_model.keras"
model.save(model_path)
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(label_encoder, 'label_encoder.joblib')
print(f"\nModel & Artefak Preprocessing sukses disimpan: {model_path}")


# 7. INFERENCE VERIFICATION (PROSES INFERENSI DATA BARU)
print("\n--- Memverifikasi Model pada Tahap Inferensi ---")
loaded_model = tf.keras.models.load_model(
    model_path,
    custom_objects={'StressFeatureExtractor': StressFeatureExtractor}
)

print("\n=== RINGKASAN ARSITEKTUR MODEL SETELAH LOAD ===")
loaded_model.summary()

real_user_example = np.array([[
    1, 4.0, 5.0, 4.0, 4.0, 5.0, 3.0, 4.0, 4.0, 4.0, 4.0, 3.0, 2.0, 2.0, 3.0, 2.0,
    50.0, 5, 30, 82, 3200, 7.5, 8.0, 2, 2
]])

new_user_scaled = scaler.transform(real_user_example)
predictions = loaded_model.predict(new_user_scaled, verbose=0)
predicted_class_idx = np.argmax(predictions, axis=1)[0]

status_mapping = {0: "HIGH STRESS", 1: "LOW STRESS", 2: "MEDIUM STRESS"}
print(f"\nHasil Prediksi: {status_mapping[predicted_class_idx]}")
print(f"Tingkat Kepercayaan Model: {np.max(predictions) * 100:.2f}%")

=== Memuat dataset dari final_stress_dataset.csv ===
[INFO] Mengonversi kolom kategorikal 'daily_pressure' ke numerik...
Jumlah Fitur (Input): 25
Jumlah Kelas (Target): 3 -> [np.int64(0), np.int64(1), np.int64(2)]

Distribusi kelas SEBELUM SMOTE (train): Counter({np.int64(2): 234, np.int64(0): 61, np.int64(1): 4})
Shape Data Train SEBELUM SMOTE: (299, 25)

[INFO] Shape Data Latih TEPAT SETELAH SMOTE: (702, 25)
Shape Data Latih SETELAH Pembulatan Diskrit: (702, 25)
Distribusi kelas SETELAH SMOTE (train): Counter({np.int64(2): 234, np.int64(0): 234, np.int64(1): 234})

=== RINGKASAN ARSITEKTUR MODEL ===


Model: "Stress_Prediction_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_25_features (InputLayer)  │ (None, 25)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ custom_feature_extractor        │ (None, 64)             │         1,664 │
│ (StressFeatureExtractor)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_dense_1 (Dense)           │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_dense_2 (Dense)           │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_dense_3 (Dense)           │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_stress_level (Dense)     │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,419 (79.76 KB)

 Trainable params: 20,419 (79.76 KB)

...

 Total params: 61,259 (239.30 KB)

 Trainable params: 20,419 (79.76 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 40,840 (159.54 KB)


Hasil Prediksi: MEDIUM STRESS
Tingkat Kepercayaan Model: 99.98%


In [ ]:
import tensorflow as tf
import numpy as np
import joblib

# KUSTOM LAYER KEPATUHAN SERIALISASI KERAS (Harus ada saat loading model dengan custom layer)
@tf.keras.utils.register_keras_serializable(package="CustomLayers")
class StressFeatureExtractor(tf.keras.layers.Layer):
    def __init__(self, units=32, **kwargs):
        super(StressFeatureExtractor, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1], self.units),
                                 initializer='random_normal', trainable=True, name='w_custom')
        self.b = self.add_weight(shape=(self.units,),
                                 initializer='zeros', trainable=True, name='b_custom')
        super(StressFeatureExtractor, self).build(input_shape)

    def call(self, inputs):
        return tf.nn.relu(tf.matmul(inputs, self.w) + self.b)

    def get_config(self):
        config = super(StressFeatureExtractor, self).get_config()
        config.update({"units": self.units})
        return config

# 1. Load Model, Scaler, dan Label Encoder yang sudah disimpan dari tahap training
# (Pastikan kamu sudah men-save scaler.pkl dan label_encoder.pkl sebelumnya)
model = tf.keras.models.load_model(
    'stress_prediction_model.keras',
    custom_objects={'StressFeatureExtractor': StressFeatureExtractor}
)
scaler = joblib.load('scaler.joblib') # Changed to .joblib as per saved file name
label_encoder = joblib.load('label_encoder.joblib') # Changed to .joblib as per saved file name

# 2. Definisikan Input Manual dari Dataset Aktual (Baris pada gambar)
manual_input = [
    0,    # gender
    2.0,  # q1_focus_difficulty
    2.0,  # q2_overthinking
    1.0,  # q3_overwhelmed
    3.0,  # q4_anxious
    3.0,  # q5_restless
    2.0,  # q6_depressed
    2.0,  # q7_sleep_problem
    1.0,  # q8_fatigue
    3.0,  # q9_headache_tension
    2.0,  # q10_responsibility_difficulty
    2.0,  # q11_irritable
    3.0,  # q12_social_withdrawal
    2.0,  # q13_control_stress
    3.0,  # q14_calm_self
    3.0,  # q15_handle_problem
    34.0, # pss_total_score
    1,    # sleep_duration / indikator tidur 1
    9,    # quality_of_sleep / indikator tidur 2
    30,   # physical_activity_level
    65,   # heart_rate
    5000, # daily_steps
    5.4,  # screen_time_hours
    3.4,  # study_hours
    0     # daily_pressure (String "Low" dikonversi ke 0)
]

# Pastikan jumlahnya 25
if len(manual_input) != 25:
    raise ValueError(f"Jumlah fitur harus 25! Saat ini ada {len(manual_input)}.")

# 3. Preprocessing Input Manual
# Ubah ke bentuk array 2D untuk scaler
input_array = np.array(manual_input).reshape(1, -1)
input_scaled = scaler.transform(input_array)

# ... (kode sebelumnya sampai bagian prediksi) ...

# 4. Melakukan Prediksi
predictions = model.predict(input_scaled)
predicted_class_index = int(np.argmax(predictions, axis=1)[0]) # Pastikan formatnya integer
confidence_score = np.max(predictions) * 100

# 5. Output Final dengan Mapping Manual ke Teks (Translator untuk UI)
# Mapping berdasarkan log training (0=High, 1=Low, 2=Medium)
stress_mapping = {
    0: "STRES TINGGI (High)",
    1: "STRES RENDAH (Low)",
    2: "STRES SEDANG (Medium)"
}

# Terjemahkan angka hasil prediksi ke dalam teks
predicted_label = stress_mapping.get(predicted_class_index, "TIDAK DIKETAHUI")

print(f"Hasil Prediksi      : {predicted_label}")
print(f"Confidence (Yakin)  : {confidence_score:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
Hasil Prediksi      : STRES SEDANG (Medium)
Confidence (Yakin)  : 100.00%
